# 11. Кластеризация и анализ ошибок правил

Минимальный эксперимент: используем признаки профилей как текстовые токены, строим кластеры и сравниваем их с текущими blocking-правилами.


In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

DATA_DIR = Path('../data/processed/er_profile_mart_multivalue')

SEED = 42
MAX_FEATURES = 50_000
SVD_COMPONENTS = 64
MAX_SAMPLE_PAIRS_PER_FAMILY = 50_000


## 1. Загружаем готовые артефакты

Берём только то, что уже построено в ноутбуке `03`: базовую таблицу профилей, multi-value слой, текущий `blocking_index` и финальный список рекомендованных правил.


In [2]:
profile_core = pd.read_parquet(DATA_DIR / 'profile_core.parquet')
profile_values = pd.read_parquet(DATA_DIR / 'profile_value_summary_long.parquet')
blocking_index = pd.read_parquet(DATA_DIR / 'blocking_index.parquet')
recommended_rules = pd.read_csv(DATA_DIR / 'recommended_blocking_rules.csv')

for df in [profile_core, profile_values, blocking_index, recommended_rules]:
    for col in ['profile_id', 'entity_id', 'block_family', 'block_rule', 'block_value', 'feature_key', 'value_norm']:
        if col in df.columns:
            df[col] = df[col].astype(str)

recommended_rule_names = set(
    recommended_rules.loc[
        recommended_rules['recommended_for_next_step'].astype(str).str.lower().eq('true'),
        'block_rule',
    ]
)
recommended_index = blocking_index[blocking_index['block_rule'].isin(recommended_rule_names)].copy()

print('profiles:', profile_core['profile_id'].nunique())
print('profile values:', len(profile_values))
print('recommended rules:', len(recommended_rule_names))


profiles: 61927
profile values: 1844644
recommended rules: 43


## 2. Строим кластеры профилей

Каждый профиль превращаем в набор токенов вида `feature_key=value`. Потом строим TF-IDF матрицу, сжимаем её через SVD и запускаем MiniBatchKMeans.


In [3]:
token_source = profile_values[
    profile_values['value_norm'].notna()
    & profile_values['feature_key'].notna()
].copy()
token_source['token'] = token_source['feature_key'] + '=' + token_source['value_norm']

profile_docs = (
    token_source.groupby('profile_id', sort=False)['token']
    .agg(lambda values: ' '.join(sorted(set(map(str, values)))))
    .rename('tokens')
    .reset_index()
)
profile_docs = profile_core[['profile_id', 'entity_id', 'entity_size']].merge(profile_docs, on='profile_id', how='left')
profile_docs['tokens'] = profile_docs['tokens'].fillna('')

n_profiles = profile_docs['profile_id'].nunique()
n_clusters = int(min(500, max(50, round(np.sqrt(n_profiles) * 2))))

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    min_df=2,
    max_df=0.8,
    max_features=MAX_FEATURES,
)
x_tfidf = vectorizer.fit_transform(profile_docs['tokens'])

n_components = min(SVD_COMPONENTS, max(2, x_tfidf.shape[1] - 1))
svd = TruncatedSVD(n_components=n_components, random_state=SEED)
x_reduced = svd.fit_transform(x_tfidf)

cluster_model = MiniBatchKMeans(
    n_clusters=n_clusters,
    random_state=SEED,
    batch_size=4096,
    n_init='auto',
)
profile_docs['cluster_id'] = cluster_model.fit_predict(x_reduced)

profile_to_cluster = profile_docs.set_index('profile_id')['cluster_id'].to_dict()

cluster_sizes = profile_docs.groupby('cluster_id')['profile_id'].nunique().rename('cluster_size').reset_index()
cluster_sizes.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T


,count,mean,std,min,50%,90%,95%,99%,max
cluster_id,498.0,248.500000,143.904482,0.0,248.5,447.3,472.15,492.03,497.0
cluster_size,498.0,124.351406,90.358770,5.0,102.5,233.6,292.60,468.72,626.0


## 3. Кластер как аналог blocking правил

Смотрим на кластеризацию так же, как на blocking-правило: если два профиля попали в один кластер, они стали парой-кандидатом.

Главный вопрос: кластеризация создаёт больше настоящих дублей, чем случайный перебор, или нет.


In [4]:
def make_positive_pairs(core: pd.DataFrame) -> pd.DataFrame:
    rows = []
    pair_id = 0
    for entity_id, grp in core.groupby('entity_id', sort=False):
        profiles = sorted(grp['profile_id'].dropna().astype(str).unique())
        if len(profiles) < 2:
            continue
        for left, right in combinations(profiles, 2):
            rows.append((pair_id, entity_id, left, right))
            pair_id += 1
    return pd.DataFrame(rows, columns=['pair_id', 'entity_id', 'profile_id_l', 'profile_id_r'])


positive_pairs = make_positive_pairs(profile_core)
positive_pairs['cluster_l'] = positive_pairs['profile_id_l'].map(profile_to_cluster)
positive_pairs['cluster_r'] = positive_pairs['profile_id_r'].map(profile_to_cluster)
positive_pairs['captured_by_cluster'] = positive_pairs['cluster_l'].eq(positive_pairs['cluster_r'])

cluster_candidate_pairs = int((cluster_sizes['cluster_size'] * (cluster_sizes['cluster_size'] - 1) // 2).sum())
cluster_positive_pairs = int(positive_pairs['captured_by_cluster'].sum())
total_positive_pairs = len(positive_pairs)
total_possible_pairs = n_profiles * (n_profiles - 1) // 2
random_positive_rate = total_positive_pairs / total_possible_pairs
cluster_positive_rate = cluster_positive_pairs / cluster_candidate_pairs

cluster_rule_summary = pd.DataFrame([
    {
        'n_clusters': n_clusters,
        'candidate_pairs': cluster_candidate_pairs,
        'positive_pairs_found': cluster_positive_pairs,
        'positive_pair_recall': cluster_positive_pairs / total_positive_pairs,
        'positive_rate_pct': cluster_positive_rate * 100,
        'random_positive_rate_pct': random_positive_rate * 100,
        'lift_vs_random': cluster_positive_rate / random_positive_rate,
        'p95_cluster_size': cluster_sizes['cluster_size'].quantile(0.95),
        'max_cluster_size': cluster_sizes['cluster_size'].max(),
    }
])
cluster_rule_summary


,n_clusters,candidate_pairs,positive_pairs_found,positive_pair_recall,positive_rate_pct,random_positive_rate_pct,lift_vs_random,p95_cluster_size,max_cluster_size
0,498,5848321,11156,0.681782,0.190756,0.000853,223.530839,292.6,626


## 4. Сравнение кластеров и рекомендованных правил на реальных дублях

Здесь не строим все пары из всех блоков. Проверяем только реальные пары дублей: для каждой такой пары смотрим, поймали её рекомендованные правила и попала ли она в один кластер.


In [5]:
profile_rule_keys = (
    recommended_index[['profile_id', 'block_rule', 'block_value']]
    .drop_duplicates()
    .assign(rule_key=lambda df: list(zip(df['block_rule'], df['block_value'])))
    .groupby('profile_id')['rule_key']
    .agg(lambda keys: set(keys))
    .to_dict()
)


def captured_by_recommended_rules(left: str, right: str) -> bool:
    return bool(profile_rule_keys.get(left, set()) & profile_rule_keys.get(right, set()))


positive_pairs['captured_by_rules'] = [
    captured_by_recommended_rules(left, right)
    for left, right in positive_pairs[['profile_id_l', 'profile_id_r']].to_numpy()
]

positive_pairs['capture_bucket'] = np.select(
    [
        positive_pairs['captured_by_rules'] & positive_pairs['captured_by_cluster'],
        positive_pairs['captured_by_rules'] & ~positive_pairs['captured_by_cluster'],
        ~positive_pairs['captured_by_rules'] & positive_pairs['captured_by_cluster'],
    ],
    ['both', 'rules_only', 'cluster_only'],
    default='missed_by_both',
)

positive_capture_summary = (
    positive_pairs['capture_bucket']
    .value_counts()
    .rename_axis('bucket')
    .reset_index(name='positive_pairs')
)
positive_capture_summary['share'] = positive_capture_summary['positive_pairs'] / len(positive_pairs)
positive_capture_summary


,bucket,positive_pairs,share
0,both,10978,0.670904
1,rules_only,3800,0.232231
2,missed_by_both,1407,0.085987
3,cluster_only,178,0.010878


## 5. Сравнение по семействам правил на сэмпле пар

Для каждого семейства правил берём ограниченный сэмпл пар-кандидатов. Смотрим, какая доля этих пар настоящая и какая доля подтверждается кластером.

Это быстрый анализ ошибок: если семейство создаёт много отрицательных пар и при этом кластер тоже часто помещает их вместе, значит оба подхода видят похожий слабый сигнал. Если кластер редко подтверждает пары семейства, семейство может быть слишком широким или шумным.


In [6]:
profile_to_entity = profile_core.set_index('profile_id')['entity_id'].to_dict()
rng = np.random.default_rng(SEED)


def sample_pairs_from_family(family_index: pd.DataFrame, max_pairs: int) -> pd.DataFrame:
    rows = []
    blocks = list(family_index[['block_rule', 'block_value', 'profile_id']].drop_duplicates().groupby(['block_rule', 'block_value'], sort=False))
    rng.shuffle(blocks)

    for (rule, block_value), grp in blocks:
        profiles = sorted(grp['profile_id'].astype(str).unique())
        if len(profiles) < 2:
            continue

        all_pair_count = len(profiles) * (len(profiles) - 1) // 2
        remaining = max_pairs - len(rows)
        if remaining <= 0:
            break

        if all_pair_count <= remaining and all_pair_count <= 2_000:
            for left, right in combinations(profiles, 2):
                rows.append((left, right, rule))
        else:
            take = min(remaining, 2_000, all_pair_count)
            seen = set()
            while len(seen) < take:
                left, right = rng.choice(profiles, size=2, replace=False)
                if left > right:
                    left, right = right, left
                seen.add((left, right))
            rows.extend((left, right, rule) for left, right in seen)

    return pd.DataFrame(rows, columns=['profile_id_l', 'profile_id_r', 'example_rule']).drop_duplicates(['profile_id_l', 'profile_id_r'])


family_rows = []
for family, family_index in recommended_index.groupby('block_family', sort=False):
    sampled_pairs = sample_pairs_from_family(family_index, MAX_SAMPLE_PAIRS_PER_FAMILY)
    if sampled_pairs.empty:
        continue

    sampled_pairs['entity_id_l'] = sampled_pairs['profile_id_l'].map(profile_to_entity)
    sampled_pairs['entity_id_r'] = sampled_pairs['profile_id_r'].map(profile_to_entity)
    sampled_pairs['is_positive_pair'] = sampled_pairs['entity_id_l'].eq(sampled_pairs['entity_id_r'])
    sampled_pairs['same_cluster'] = sampled_pairs['profile_id_l'].map(profile_to_cluster).eq(
        sampled_pairs['profile_id_r'].map(profile_to_cluster)
    )

    positives = sampled_pairs[sampled_pairs['is_positive_pair']]
    negatives = sampled_pairs[~sampled_pairs['is_positive_pair']]
    family_rows.append({
        'block_family': family,
        'sampled_pairs': len(sampled_pairs),
        'positive_rate_pct': sampled_pairs['is_positive_pair'].mean() * 100,
        'cluster_agreement_pct': sampled_pairs['same_cluster'].mean() * 100,
        'positive_cluster_agreement_pct': positives['same_cluster'].mean() * 100 if len(positives) else np.nan,
        'negative_cluster_agreement_pct': negatives['same_cluster'].mean() * 100 if len(negatives) else np.nan,
    })

family_cluster_error_summary = pd.DataFrame(family_rows).sort_values(
    ['positive_rate_pct', 'cluster_agreement_pct'],
    ascending=[False, False],
)
family_cluster_error_summary


,block_family,sampled_pairs,positive_rate_pct,cluster_agreement_pct,positive_cluster_agreement_pct,negative_cluster_agreement_pct
9,identity_rescue,78,100.000000,41.025641,41.025641,NaN
7,registration_time_window,20246,35.597155,48.108268,91.147495,24.319350
6,behavior_daypart_device,45471,1.796750,14.290427,59.485924,13.463520
5,behavior_daypart,44079,1.567640,20.079857,63.675832,19.385544
2,behavior_context,47144,1.465722,25.002121,56.295224,24.536628
4,behavior_context_device,45677,1.394575,11.977582,51.962323,11.412078
3,postman_context,47794,0.608863,22.458886,58.419244,22.238595
8,cluster_candidate,42322,0.489107,4.560276,57.971014,4.297756
0,context,49988,0.402097,11.826838,49.253731,11.675739
1,behavior,49935,0.382497,2.583358,55.497382,2.380187


## 6. Какие признаки пропускают правила

Разбираем `cluster_only`: это реальные дубли, которые кластеризация поймала, а рекомендованные правила не
поймали.

Сравниваем их с `missed_by_both`: реальными дублями, которые не нашли ни правила, ни кластеризация. Если признак заметно чаще совпадает в `cluster_only`, значит кластеризация использовала сигнал, который текущие правила слабо покрывают.


In [ ]:
from collections import Counter


profile_feature_values = (
    profile_values[['profile_id', 'feature_key', 'value_norm']]
    .dropna()
    .groupby(['profile_id', 'feature_key'])['value_norm']
    .agg(lambda values: set(map(str, values)))
    .reset_index()
)

profile_feature_map = {}
for row in profile_feature_values.itertuples(index=False):
    profile_feature_map.setdefault(row.profile_id, {})[row.feature_key] = row.value_norm


def shared_feature_stats(pairs: pd.DataFrame, label: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_counter = Counter()
    token_counter = Counter()

    for left, right in pairs[['profile_id_l', 'profile_id_r']].itertuples(index=False):
        left_features = profile_feature_map.get(left, {})
        right_features = profile_feature_map.get(right, {})

        for feature_key in set(left_features) & set(right_features):
            shared_values = left_features[feature_key] & right_features[feature_key]
            if not shared_values:
                continue

            feature_counter[feature_key] += 1
            for value_norm in shared_values:
                token_counter[(feature_key, value_norm)] += 1

    n_pairs = len(pairs)
    feature_stats = pd.DataFrame([
        {
            'feature_key': feature_key,
            f'{label}_pair_count': pair_count,
            f'{label}_share': pair_count / n_pairs if n_pairs else np.nan,
        }
        for feature_key, pair_count in feature_counter.items()
    ])

    token_stats = pd.DataFrame([
        {
            'feature_key': feature_key,
            'value_norm': value_norm,
            f'{label}_pair_count': pair_count,
            f'{label}_share': pair_count / n_pairs if n_pairs else np.nan,
        }
        for (feature_key, value_norm), pair_count in token_counter.items()
    ])
    return feature_stats, token_stats


cluster_only_pairs = positive_pairs[positive_pairs['capture_bucket'].eq('cluster_only')].copy()
missed_by_both_pairs = positive_pairs[positive_pairs['capture_bucket'].eq('missed_by_both')].copy()
all_positive_pairs = positive_pairs.copy()

cluster_only_features, cluster_only_tokens = shared_feature_stats(cluster_only_pairs, 'cluster_only')
missed_by_both_features, _ = shared_feature_stats(missed_by_both_pairs, 'missed_by_both')
all_positive_features, _ = shared_feature_stats(all_positive_pairs, 'all_positive')

missed_feature_candidates = (
    cluster_only_features
    .merge(missed_by_both_features, on='feature_key', how='left')
    .merge(all_positive_features, on='feature_key', how='left')
)

for col in ['missed_by_both_pair_count', 'missed_by_both_share', 'all_positive_pair_count', 'all_positive_share']:
    missed_feature_candidates[col] = missed_feature_candidates[col].fillna(0)

missed_feature_candidates['lift_vs_missed_by_both'] = (
    missed_feature_candidates['cluster_only_share']
    / missed_feature_candidates['missed_by_both_share'].replace(0, np.nan)
)
missed_feature_candidates['lift_vs_all_positive'] = (
    missed_feature_candidates['cluster_only_share']
    / missed_feature_candidates['all_positive_share'].replace(0, np.nan)
)

missed_feature_candidates = missed_feature_candidates.sort_values(
    ['cluster_only_share', 'lift_vs_missed_by_both'],
    ascending=[False, False],
)

missed_feature_candidates.head(20)


In [ ]:
cluster_only_tokens.sort_values('cluster_only_pair_count', ascending=False).head(20)


### Короткий вывод

Анализ результатов кластеризации показал: среди реальных совпадений, пропущенных текущими blocking-правилами, часто встречаются np__is_not_russia, browser, device, os, postman и короткие поведенческие признаки fs. Это указывает, что кластеризация находит часть клиентов по похожему устройству и поведению даже без совпадения точного города или региона. Такие признаки нельзя сразу добавлять как отдельные широкие правила из-за риска ошибочных объединений; их следует проверять только в составе узких комбинаций признаков.
